<a href="https://colab.research.google.com/github/Otza02/satellite-img-segmentation/blob/main/notebooks/train-unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Otza02/satellite-img-segmentation.git
%cd satellite-img-segmentation
%pip install -e .

Cloning into 'satellite-img-segmentation'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 247 (delta 110), reused 194 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (247/247), 1.58 MiB | 18.59 MiB/s, done.
Resolving deltas: 100% (110/110), done.
/content/satellite-img-segmentation
Obtaining file:///content/satellite-img-segmentation
  Preparing metadata (setup.py) ... done
  Running setup.py develop for satelliteSegmentation


## Reiniciar sesion

In [2]:
%cd /content/satellite-img-segmentation
!unzip -q data/train.zip -d data/train

/content/satellite-img-segmentation


## Entrenamiento de U-Net

In [3]:
from satelliteSegmentation.config import Config
from satelliteSegmentation.dataset import SatelliteData, spatial_train_val_split
from satelliteSegmentation.models.unet import UNet
from satelliteSegmentation.train import train_model
from satelliteSegmentation.metrics import segmentation_metrics, dice_score
from satelliteSegmentation.utils import plot_confusion_matrix, plot_bar_metrics
from satelliteSegmentation.tokenizer import Tokenizer

import torch
from torch.utils.data import DataLoader, Subset
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
import json
import shutil

from google.colab import files

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

## Preparar Datos

In [5]:
data = SatelliteData("data/train")

train_idx, val_idx = spatial_train_val_split(data, val_fraction=0.2)
train, val = Subset(data, train_idx), Subset(data, val_idx)

train_loader = DataLoader(
    train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=device == "cuda",
    persistent_workers=device == "cuda",
)

val_loader = DataLoader(
    val,
    batch_size=batch_size,
    num_workers=2,
    pin_memory=device == "cuda",
    persistent_workers=device == "cuda",
)

100%|██████████| 5184/5184 [00:11<00:00, 466.85it/s]


Dataset cargado:
X shape = torch.Size([5184, 3, 120, 120])
Y shape = torch.Size([5184, 120, 120])


In [6]:
def calc_weights(
    dataloader,
    num_classes: int,
    ignore_index: int,
):
    counts = torch.zeros(num_classes, dtype=torch.long)

    for _, masks in dataloader:
        masks = masks.flatten()
        masks = masks[masks != ignore_index]

        counts += torch.bincount(
            masks,
            minlength=num_classes,
        )

    # Solo clases que participan en el entrenamiento
    valid_counts = counts.clone()
    valid_counts[ignore_index] = 0

    # Ejemplo: inverse square root frequency
    weights = torch.zeros(num_classes, dtype=torch.float32)

    valid = valid_counts > 0
    weights[valid] = 1.0 / torch.sqrt(valid_counts[valid].float())

    # Normalizar para que la media de los weights válidos sea 1
    weights[valid] /= weights[valid].mean()

    # El weight de la clase ignorada es irrelevante
    weights[ignore_index] = 0.0

    return counts, weights

counts, weights = calc_weights(
    train_loader,
    num_classes=7,
    ignore_index=6,
)
weights = weights.to(device)

In [7]:
def save_model(name: str, model: torch.nn.Module, hist: dict, config: Config):
    result_folder = Path(f"result/{name}")
    result_folder.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), result_folder / "checkpoint.pt")
    pd.DataFrame(hist).to_csv(result_folder / "hist.csv", index=False)
    with open(result_folder / "config.json", "w") as f:
        json.dump(config.to_json(), f)

    return result_folder

def zip_and_download(folder: Path):
    folder = Path(folder)

    zip_path = shutil.make_archive(
        base_name=str(folder),
        format="zip",
        root_dir=folder.parent,
        base_dir=folder.name,
    )

    print(f"Descargando: {zip_path}")
    files.download(zip_path)

## Train

In [8]:
configs = {
    "baseline": Config(device, weights=weights),
    "channels-32": Config(device, hidden_channels=(32, 64, 128, 256), bottleneck_channels=512, weights=weights),
    "kernel-5": Config(device, kernel_size=5, weights=weights),
    "lr-3e-4": Config(device, lr=3e-4, weights=weights),
    "lr-1e-5": Config(device, lr=1e-5, weights=weights),
}

In [9]:
for name, conf in configs.items():
    torch.manual_seed(2026)
    model = UNet(conf)
    print(f"Modelo: {name} | n_params: {sum([p.numel() for p in model.parameters()]):,}")

    criterion = torch.nn.CrossEntropyLoss(conf.weights, ignore_index=6)

    model, hist = train_model(model, train_loader, val_loader, criterion, conf)

    result_folder = save_model(name, model, hist, conf)
    zip_and_download(result_folder)

Modelo: baseline | n_params: 31,043,911
Epoch 01/100 | train_loss=1.2051 | val_loss=0.9085 | time 00:25
Epoch 02/100 | train_loss=0.7982 | val_loss=0.9226 | time 00:24
Epoch 03/100 | train_loss=0.6998 | val_loss=0.7516 | time 00:24
Epoch 04/100 | train_loss=0.6087 | val_loss=0.5580 | time 00:25
Epoch 05/100 | train_loss=0.5789 | val_loss=0.5196 | time 00:25
Epoch 06/100 | train_loss=0.5227 | val_loss=0.5295 | time 00:26
Epoch 07/100 | train_loss=0.4789 | val_loss=0.4717 | time 00:26
Epoch 08/100 | train_loss=0.4425 | val_loss=0.6732 | time 00:25
Epoch 09/100 | train_loss=0.4368 | val_loss=0.7389 | time 00:26
Epoch 10/100 | train_loss=0.3941 | val_loss=0.6662 | time 00:26
Epoch 11/100 | train_loss=0.3797 | val_loss=0.4902 | time 00:26
Epoch 12/100 | train_loss=0.3644 | val_loss=0.4508 | time 00:25
Epoch 13/100 | train_loss=0.3310 | val_loss=0.4369 | time 00:26
Epoch 14/100 | train_loss=0.3128 | val_loss=0.4592 | time 00:26
Epoch 15/100 | train_loss=0.3203 | val_loss=0.3720 | time 00:26


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Modelo: channels-32 | n_params: 7,766,183
Epoch 01/100 | train_loss=1.5223 | val_loss=1.2452 | time 00:10
Epoch 02/100 | train_loss=1.1233 | val_loss=0.8604 | time 00:10
Epoch 03/100 | train_loss=0.9635 | val_loss=0.8669 | time 00:10
Epoch 04/100 | train_loss=0.8629 | val_loss=0.8269 | time 00:11
Epoch 05/100 | train_loss=0.7859 | val_loss=0.7113 | time 00:10
Epoch 06/100 | train_loss=0.7296 | val_loss=0.7088 | time 00:10
Epoch 07/100 | train_loss=0.6760 | val_loss=0.6225 | time 00:10
Epoch 08/100 | train_loss=0.6239 | val_loss=0.5961 | time 00:10
Epoch 09/100 | train_loss=0.5766 | val_loss=0.5427 | time 00:10
Epoch 10/100 | train_loss=0.5441 | val_loss=0.5178 | time 00:10
Epoch 11/100 | train_loss=0.5128 | val_loss=0.5083 | time 00:10
Epoch 12/100 | train_loss=0.4759 | val_loss=0.5518 | time 00:10
Epoch 13/100 | train_loss=0.4427 | val_loss=0.4990 | time 00:10
Epoch 14/100 | train_loss=0.4306 | val_loss=0.4503 | time 00:10
Epoch 15/100 | train_loss=0.3978 | val_loss=0.4348 | time 00:1

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Modelo: lr-1e-5 | n_params: 31,043,911
Epoch 01/100 | train_loss=1.7688 | val_loss=1.5699 | time 00:26
Epoch 02/100 | train_loss=1.3905 | val_loss=1.1812 | time 00:26
Epoch 03/100 | train_loss=1.1301 | val_loss=0.9913 | time 00:26
Epoch 04/100 | train_loss=0.9849 | val_loss=1.0302 | time 00:26
Epoch 05/100 | train_loss=0.8851 | val_loss=0.7861 | time 00:26
Epoch 06/100 | train_loss=0.8196 | val_loss=0.7265 | time 00:26
Epoch 07/100 | train_loss=0.7705 | val_loss=0.7054 | time 00:26
Epoch 08/100 | train_loss=0.7154 | val_loss=0.7322 | time 00:26
Epoch 09/100 | train_loss=0.6873 | val_loss=0.6736 | time 00:26
Epoch 10/100 | train_loss=0.6684 | val_loss=0.7522 | time 00:26
Epoch 11/100 | train_loss=0.6237 | val_loss=0.6637 | time 00:26
Epoch 12/100 | train_loss=0.6046 | val_loss=0.6612 | time 00:26
Epoch 13/100 | train_loss=0.5799 | val_loss=0.6554 | time 00:26
Epoch 14/100 | train_loss=0.5611 | val_loss=0.8151 | time 00:26
Epoch 15/100 | train_loss=0.5370 | val_loss=0.6693 | time 00:26
E

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>